In [4]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import scipy
import torch
import torch.nn as nn

import pymc
from magnus import NSFeas
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "plots" / "tools"))
from trellis import trellis_plot #pyright:ignore
from corner import corner_plot #pyright:ignore

In [5]:
# Helpers
def theta_nominal():
    T_REF = 328.15
    k_1_ref = 8.84e-3 * 60 * 0.5 # 1/min
    k_2_ref = k_1_ref / 8
    theta_1 = k_1_ref * np.exp(E_1 / (R_GAS * T_REF))
    theta_2 = k_2_ref * np.exp(E_2 / (R_GAS * T_REF))

    return theta_1, theta_2

In [6]:
# Global Constants
E_1 = 48_000 # J/mol (activation energy first reaction)
E_2 = 55_000 # J/mol (activation energy second reaction)
R_GAS = 8.314 # J/K/mol
th_1, th_2 = theta_nominal()
BENZENE_FEED_CONC = 11.28e3 # mol/m3

In [7]:
# Calculation of arrhenius prefactors (T-dependent)

def compute_k(T_1, T_2):
    k_1_R1 = th_1 * np.exp(- E_1 / (R_GAS * T_1))
    k_2_R1 = th_2 * np.exp(- E_2 / (R_GAS * T_1))
    k_1_R2 = th_1 * np.exp(- E_1 / (R_GAS * T_2))
    k_2_R2 = th_2 * np.exp(- E_2 / (R_GAS * T_2))

    return (k_1_R1, k_2_R1), (k_1_R2, k_2_R2)

In [8]:
# Counters (for stat purposes only)
M1_counter = R1_counter = R2_counter = S1_counter = 0

In [9]:
# Mixer
def M1(rho, c_R, c_F):
    global M1_counter; M1_counter += 1
    c_A_R, c_B_R, c_C_R = c_R[0], c_R[1], c_R[2]
    c_A_F, c_B_F, c_C_F = c_F[0], c_F[1], c_F[2]
    c_A_alpha = rho * c_A_R + (1 - rho) * c_A_F
    c_B_alpha = rho * c_B_R + (1 - rho) * c_B_F
    c_C_alpha = rho * c_C_R + (1 - rho) * c_C_F

    return c_A_alpha, c_B_alpha, c_C_alpha

In [10]:
# CSTR1
def R1(c_alpha, k_R1, tau_1):
    global R1_counter; R1_counter += 1
    k_1_R1, k_2_R1 = k_R1[0], k_R1[1]
    c_A_alpha, c_B_alpha, c_C_alpha = c_alpha[0], c_alpha[1], c_alpha[2]

    c_A_R1 = c_A_alpha / (1 + k_1_R1 * tau_1)
    c_B_R1 = (c_B_alpha + k_1_R1 * c_A_R1 * tau_1) / (1 + k_2_R1 * tau_1)
    c_C_R1 = c_C_alpha + k_2_R1 * c_B_R1 * tau_1
    
    return  c_A_R1, c_B_R1, c_C_R1

In [11]:
# CSTR2
def R2(c_R1, k_R2, tau_2):
    global R2_counter; R2_counter += 1
    c_A_R1, c_B_R1, c_C_R1 = c_R1[0], c_R1[1], c_R1[2]
    k_1_R2, k_2_R2 = k_R2[0], k_R2[1]

    c_A_R2 = c_A_R1 / (1 + k_1_R2 * tau_2)
    c_B_R2 = (c_B_R1 + k_1_R2 * c_A_R2 * tau_2) / (1 + k_2_R2 * tau_2)
    c_C_R2 = c_C_R1 + k_2_R2 * c_B_R2 * tau_2

    return c_A_R2, c_B_R2, c_C_R2

In [12]:
# Splitter
def S1(c_R2):
    # NOTE: rho: recycle ratio, dens: density
    global S1_counter; S1_counter += 1
    c_A_R2, c_B_R2, c_C_R2 = c_R2[0], c_R2[1], c_R2[2]
    M_A, M_B, M_C = 78.11, 112.56, 147.00 # g/mol
    dens_A, dens_B, dens_C = 0.88e6, 1.11e6, 1.30e6 # g/m3
    s_A, s_B, s_C = 0.7, 0.35, 0.20 # these values are for illustration
    summands = [M_A / dens_A  * s_A * c_A_R2, M_B / dens_B  * s_B * c_B_R2, M_C / dens_C  * s_C * c_C_R2]
    rho = sum(summands)
    c_A_R = s_A * c_A_R2 / rho
    c_B_R = s_B * c_B_R2 / rho
    c_C_R = s_C * c_C_R2 / rho

    return (c_A_R, c_B_R, c_C_R), rho

In [13]:
def get_informed_guess():
    """Make informed initial guess by finding nearest feasible configuration
    First value: is informed guess available (NONE if no feasible points found yet),
    Second value: rho_ig,
    Third value: c_R_ig
    TODO: implement this if fixed point iterations are slow
    """
    return None, None, None

In [14]:
def is_converged(c_R_real_guess, rho_real_guess):
    """Takes in tuples of real and guessed values for c_R 
    (recycle stream concentration), and rho (recycle ratio)
    and compares them."""
    (c_A_R_real, c_B_R_real, c_C_R_real), (c_A_R_guess, c_B_R_guess, c_C_R_guess) = c_R_real_guess
    rho_real, rho_guess = rho_real_guess
    real = np.array([c_A_R_real, c_B_R_real, c_C_R_real, rho_real])
    guess = np.array([c_A_R_guess, c_B_R_guess, c_C_R_guess, rho_guess])
    diff = np.abs(real - guess) # abs is safeguarding only
    diff_norm = diff / max((abs(real)).max(), 1e-8)
    max_vio = diff_norm.max()

    return max_vio < 1e-8

In [15]:
class SimResult:
    def __init__(self, converged, c_alpha, c_R1, c_R2, c_R, rho, c_F):
        self.converged = converged
        self.c_alpha = c_alpha
        self.c_R1 = c_R1
        self.c_R2 = c_R2
        self.c_R = c_R
        self.c_F = c_F
        self.rho = rho

In [16]:
# Simulate forward (to be called by Nested Sampler)
def sim_forward(T_1, tau_1, T_2, tau_2):
    it, MAX_ITER = 0, 1000
    c_F = BENZENE_FEED_CONC, 0, 0 # mol/m3 (pure benzene feed)
    # initial guesses (ig)
    inf_guess = get_informed_guess()
    rho_ig = 0.3 if not inf_guess[0] else inf_guess[1]
    c_R_ig = (5, 2, 1) if not inf_guess[0] else inf_guess[2]

    # k need fresh computatation at each configuration (T-dependent)
    k_R1, k_R2 = compute_k(T_1, T_2)

    while (it < MAX_ITER):
        it += 1
        c_alpha = M1(rho=rho_ig, c_R=c_R_ig, c_F=c_F)
        c_R1 = R1(c_alpha=c_alpha, k_R1=k_R1, tau_1=tau_1)
        c_R2 = R2(c_R1=c_R1, k_R2=k_R2, tau_2=tau_2)
        c_R, rho = S1(c_R2=c_R2)
        conv = is_converged((c_R, c_R_ig), (rho, rho_ig))
        if conv:
            # NOTE: return guess not result
            return SimResult(converged=True, c_alpha=c_alpha,
                             c_R1=c_R1, c_R2=c_R2,
                             c_R=c_R, c_F=c_F, rho=rho)

        # update to next tear iteration
        c_R_ig, rho_ig = c_R, rho

    # not converged within MAX_ITER
    return SimResult(False, None, None, None, None, None, None)

In [17]:
# Interface Adapter
def model_simultaneous(x):
    T_1, tau_1, T_2, tau_2 = x[0], x[1], x[2], x[3]
    sim_res = sim_forward(T_1=T_1, tau_1=tau_1, T_2=T_2, tau_2=tau_2)

    if not sim_res.converged or sim_res.rho < 0 or sim_res.rho > 1:
        # return *something* guaranteed to be infeasible
        return [-1e-8, -1e-8, -1e-8, -1e-8, -1e-8, -1e-8]
    c_A_R1, c_B_R1, c_C_R1 = sim_res.c_R1[0], sim_res.c_R1[1], sim_res.c_R1[2] 
    c_A_R2, c_B_R2, c_C_R2 = sim_res.c_R2[0], sim_res.c_R2[1], sim_res.c_R2[2] 

    return [c_A_R1, c_A_R2, c_B_R1, c_B_R2, c_C_R1, c_C_R2]

In [18]:
def simultaneous_NS():
    """Performs nested sampling on entire search space"""
    # Nested Sampling of [T1, tau1, T2, tau2] space
    DAG = pymc.FFGraph()
    DAG.options.MAXTHREAD = 1

    T_1, tau_1 = DAG.add_var('T_1'), DAG.add_var('tau_1')
    T_2, tau_2 = DAG.add_var('T_2'), DAG.add_var('tau_2')

    op = pymc.FFCustom()
    op.set_D_eval(model_simultaneous)
    # applying the operation adds nodes to DAG
    outs = op(ndep=6, var=[T_1, tau_1, T_2, tau_2], uid=0)
    c_A_R1, c_A_R2, c_B_R1, c_B_R2, c_C_R1, c_C_R2 = outs[0], outs[1], outs[2], outs[3], outs[4], outs[5]

    # Setup nested sampler
    NS = NSFeas()
    NS.options.MAXTHREAD = 1           # Python callback + GIL: must be 1
    NS.options.DISPLEVEL = 1           # console verbosity
    NS.options.DISPITER  = 25          # rows per N iterations in table (cosmetic)
    NS.options.NUMLIVE   = 8192        # #Live Points
    NS.options.NUMPROP   = 256         # Proposals per iteration
    NS.options.MAXITER   = 0           # iterate until complete (hard iteration limit else)

    NS.set_dag(DAG)
    # contols for variables and bounds
    NS.set_control([T_1, tau_1, T_2, tau_2], [298.15, 2, 298.15, 2], [353.15, 30, 353.15, 30])
    NS.set_constraint([c_C_R1 / (c_A_R1 + c_B_R1 + c_C_R1) - 0.05,
                    0.55 - c_B_R2 / (c_A_R2 + c_B_R2 + c_C_R2), 
                    0.70 - (1 - c_A_R2 / BENZENE_FEED_CONC)])
    NS.setup()
    NS.sample()
    NS.stats.display()
    assert M1_counter == R1_counter == R2_counter == S1_counter and M1_counter != 0
    return NS

# NS_simultaneous = simultaneous_NS()

In [19]:
def simultaneous_trellis_plot():
    # NS.livepoints returns: (points[N, nu],  crit[N, 1],  prob[N, 1],  aux[N, 1])
    live_pts, live_feas, *_ = NS_simultaneous.live_points
    dead_pts, dead_feas, *_ = NS_simultaneous.dead_points

    fig = trellis_plot(
        [dead_pts[:, 0], live_pts[:, 0]],   # T_1
        [dead_pts[:, 2], live_pts[:, 2]],   # T_2
        [dead_pts[:, 1], live_pts[:, 1]],   # tau_1
        [dead_pts[:, 3], live_pts[:, 3]],   # tau_2
        x_label="$T_1$ [K]",
        y_label=r"$T_2$ [K]",
        col_label=r"$\tau_1$ [min]",
        row_label=r"$\tau_2$ [min]",
        colors=["crimson", "royalblue"],
        labels=["dead", "live"],
    )

# simultaneous_trellis_plot()

In [20]:
def simultaneous_corner_plot():
    live_pts, live_feas, *_ = NS_simultaneous.live_points
    dead_pts, dead_feas, *_ = NS_simultaneous.dead_points
    fig = corner_plot(
        [  # painter's order: dead below, live (feasible) on top
            dict(data=dead_pts, color="crimson", ms=2, alpha=0.5, label=f"dead ({len(dead_pts)})"),
            dict(data=live_pts, color="royalblue", ms=2, alpha=0.5, label=f"live ({len(live_pts)})"),
        ],
        labels=[r"$T_1$ [K]", r"$\tau_1$ [min]", r"$T_2$ [K]", r"$\tau_2$ [min]"],
        bounds=[(298.15, 353.15), (2, 30), (298.15, 353.15), (2, 30)],
        legend=True,
        title=r"Recycle cascade ($\rho = 0$): live vs dead points",
    )

# simultaneous_corner_plot()

# Decomposition based sampling

### Inlet Bounds Generation

In [21]:
def generate_sobol_points(m: int, bounds):
    """
    m: exponent to generate 2^m points
    """
    sbsam = scipy.stats.qmc.Sobol(d=4, seed=42)
    rand_norm = sbsam.random_base2(m=m) # 2^m points
    rand = rand_norm * (bounds[1] - bounds[0]) + bounds[0]
    return rand

In [22]:
def initialize_search_space():
    k_bounds = np.array([[298.15, 2, 298.15, 2], [353.15, 30, 353.15, 30]]) # T1, tau1, T2, tau2
    v_train = generate_sobol_points(6, k_bounds) # 64 points > 40 == 10 * dim(search space)
    results = {}
    for point in v_train:
        T_1, tau_1, T_2, tau_2 = point[0], point[1], point[2], point[3]
        results[tuple(point)] = sim_forward(T_1=T_1, tau_1=tau_1, T_2=T_2, tau_2=tau_2)
    return results
initialization_results = initialize_search_space()

In [23]:
# bounds for KU_i
class UnitOpBounds:
    def __init__(self, unit_name):
        self.unit_name = unit_name
        self.bounds = {}
    def add(self, var: str):
        self.bounds[var] = [float('inf'), float('-inf')] # lb, ub
    def update(self, var, val):
        assert var in self.bounds
        self.bounds[var] = [min(self.bounds[var][0], val), max(self.bounds[var][1], val)]

In [24]:
# Note: m1 has four inputs
m1_init_bounds = UnitOpBounds('M1')
m1_init_bounds.add('rho')
m1_init_bounds.add('c_A_R')
m1_init_bounds.add('c_B_R')
m1_init_bounds.add('c_C_R')

r1_init_bounds = UnitOpBounds('R1')
r1_init_bounds.add('c_A_alpha')
r1_init_bounds.add('c_B_alpha')
r1_init_bounds.add('c_C_alpha')

r2_init_bounds = UnitOpBounds('R2')
r2_init_bounds.add('c_A_R1')
r2_init_bounds.add('c_B_R1')
r2_init_bounds.add('c_C_R1')

s1_init_bounds = UnitOpBounds('S1')
s1_init_bounds.add('c_A_R2')
s1_init_bounds.add('c_B_R2')
s1_init_bounds.add('c_C_R2')

In [25]:
for v in initialization_results.values():
    m1_init_bounds.update('c_A_R', v.c_R[0])
    m1_init_bounds.update('c_B_R', v.c_R[1])
    m1_init_bounds.update('c_C_R', v.c_R[2])
    m1_init_bounds.update('rho', v.rho)

    r1_init_bounds.update('c_A_alpha', v.c_alpha[0])
    r1_init_bounds.update('c_B_alpha', v.c_alpha[1])
    r1_init_bounds.update('c_C_alpha', v.c_alpha[2])

    r2_init_bounds.update('c_A_R1', v.c_R1[0])
    r2_init_bounds.update('c_B_R1', v.c_R1[1])
    r2_init_bounds.update('c_C_R1', v.c_R1[2])

    s1_init_bounds.update('c_A_R2', v.c_R2[0])
    s1_init_bounds.update('c_B_R2', v.c_R2[1])
    s1_init_bounds.update('c_C_R2', v.c_R2[2])

### Unit Op ANN training template

In [26]:
# ANN training template
class Surrogate:
    """Plain-numpy MLP: scaling folded into first/last layer"""
    def __init__(self, W, b):
        self.W, self.b = W, b

    def __call__(self, x):
        x = np.asarray(x, dtype=float)
        single = x.ndim == 1
        h = np.atleast_2d(x)
        for i, (Wl, bl) in enumerate(zip(self.W, self.b)):
            h = h @ Wl.T + bl
            if i + 1 < len(self.W):
                h = np.tanh(h)
        return h[0] if single else h

    def export(self, path):          # optional: mlp_ffvar.hpp's format
        with open(path, "w") as f:
            f.write(f"{len(self.W)}\n")
            for Wl, bl in zip(self.W, self.b):
                f.write(f"{Wl.shape[1]} {Wl.shape[0]}\n")
                for row in Wl:
                    f.write(" ".join(f"{v:.17g}" for v in row) + "\n")
                f.write(" ".join(f"{v:.17g}" for v in bl) + "\n")

In [27]:
def fit_surrogate(X, Y, hid=16, epochs=3000, lr=1e-3, seed=0, verbose=True):
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    if Y.ndim == 1:
        Y = Y[:, None]                       # tolerate 1-D targets
    torch.manual_seed(seed)

    xmu, xsig = X.mean(0), X.std(0) + 1e-12
    ymu, ysig = Y.mean(0), Y.std(0) + 1e-12
    tx = torch.tensor((X - xmu) / xsig, dtype=torch.float64)
    ty = torch.tensor((Y - ymu) / ysig, dtype=torch.float64)

    net = nn.Sequential(
        nn.Linear(X.shape[1], hid), nn.Tanh(),
        nn.Linear(hid, hid), nn.Tanh(),
        nn.Linear(hid, Y.shape[1]),
    ).double()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(net(tx), ty)
        loss.backward()
        opt.step()
        if verbose and ep % 500 == 0:
            print(f"  epoch {ep:5d} mse(std) {loss.item():.3e}")

    # fold standardization into affine layers -> raw-unit in, raw-unit out
    lins = [m for m in net if isinstance(m, nn.Linear)]
    W = [m.weight.detach().numpy().copy() for m in lins]
    b = [m.bias.detach().numpy().copy() for m in lins]
    b[0] = b[0] - W[0] @ (xmu / xsig)
    W[0] = W[0] / xsig[None, :]
    W[-1] = ysig[:, None] * W[-1]
    b[-1] = ysig * b[-1] + ymu
    return Surrogate(W, b)

In [28]:
# TODO:
# replacement of all four unit operations with surrogates
# nested sample from inflated search space by 15% each direction && classic v_i parameter bounds if available
# use surrogates to perform global nested sampling
# find appropriate bounds for KU_i (not V_i)
# for unit in units:
#   nested sample the unit's (v_i + u_i) -> y_i => G_i <= 0 ? => classify
#   train ANN for unit mapping input -> output
# Do same as above solving the flowsheet over many different configs (suggested by Nested sampling),
# however replacing each unit with it's respective surrogate, thus solving a "surrogate flowsheet"

### Nested Sampling Per Unit

In [29]:
def nested_sample_unit(box, unit_model, n_out, make_constraints,
                       uid=1, n_live=4096, n_prop=256):
    """Nested-sample one unit over `box` and record ALL evaluations.
    box:              dict var_name -> [lb, ub]; ORDER defines x indexing!
    unit_model:       function x -> list of n_out outputs (plain floats)
    make_constraints: function outs (FFVars) -> list of constraints (feasible <= 0)
    uid:              unique per FFCustom op in this kernel session
    Returns X (n, dim_in), Y (n, n_out) numpy arrays of every evaluation."""
    records_x = []
    records_y = []

    def recording_model(x):
        y = unit_model(x)
        records_x.append(list(x))
        records_y.append(list(y))
        return y

    var_names, lb, ub = [], [], []
    for vname, bnd in box.items():
        var_names.append(vname)
        lb.append(bnd[0])
        ub.append(bnd[1])

    DAG = pymc.FFGraph()
    DAG.options.MAXTHREAD = 1
    ffvars = []
    for vname in var_names:
        ffvars.append(DAG.add_var(vname))

    op = pymc.FFCustom()
    op.set_D_eval(recording_model)
    outs = op(ndep=n_out, var=ffvars, uid=uid)

    NS = NSFeas()
    NS.options.MAXTHREAD = 1       # Python callback + GIL
    NS.options.DISPLEVEL = 1
    NS.options.DISPITER  = 25
    NS.options.NUMLIVE   = n_live
    NS.options.NUMPROP   = n_prop
    NS.options.MAXITER   = 0
    NS.set_dag(DAG)
    NS.set_control(ffvars, lb, ub)
    NS.set_constraint(make_constraints(outs))
    NS.setup()
    NS.sample()
    NS.stats.display()

    return np.array(records_x), np.array(records_y)

In [31]:
# Modeling R1

def R1_NS_wrap(x):
    T_1, tau_1, c_A_alpha, c_B_alpha, c_C_alpha = x
    k_R1 = compute_k(T_1, 42)[0]
    return R1(c_alpha=(c_A_alpha, c_B_alpha, c_C_alpha), k_R1=k_R1, tau_1=tau_1)

nested_sample_unit(
    box={'T_1': [298.15, 353.15], 'tau_1': [2, 30], **r1_init_bounds.bounds},
    unit_model=R1_NS_wrap,
    n_out=3,
    # g1 constraint
    make_constraints=lambda outs: outs[2] / (outs[0] + outs[1] + outs[2]) - 0.05,
    uid=42, # TODO: generate
    n_live=8192,
    n_prop=256
)


** INITIALIZING LIVE POINTS (8192)      0.03 SEC

Iterate        Contour #Feas    #Dead         Factor
----------------------------------------------------
      0     7.6004e-01    71        0     3.0000e-01
     25     2.8531e-01   128     4726     2.6731e-01
     50     1.8608e-01   184     7698     2.4861e-01
     75     1.4876e-01   243     9859     2.3583e-01
    100     1.2819e-01   300    11582     2.2611e-01
    125     1.1467e-01   355    12990     2.1847e-01
    150     1.0420e-01   407    14195     2.1214e-01
    175     9.5751e-02   466    15248     2.0676e-01
    200     8.8198e-02   533    16250     2.0176e-01
    225     8.1639e-02   603    17207     1.9710e-01
    250     7.5012e-02   680    18174     1.9250e-01
    275     6.9076e-02   761    19102     1.8819e-01
    300     6.4020e-02   849    19993     1.8414e-01
    325     5.9384e-02   938    20838     1.8038e-01
    350     5.4846e-02  1045    21684     1.7669e-01
    375     5.0963e-02  1150    22472     1.7333

(array([[3.25650000e+02, 1.60000000e+01, 8.18561097e+03, 1.40648387e+03,
         1.06118348e+03],
        [3.39400000e+02, 9.00000000e+00, 8.00121237e+03, 8.45585275e+02,
         1.51629308e+03],
        [3.11900000e+02, 2.30000000e+01, 8.37000956e+03, 1.96738246e+03,
         6.06073877e+02],
        ...,
        [3.15329802e+02, 3.68354803e+00, 8.23682873e+03, 1.13306121e+03,
         4.20142271e+02],
        [2.98354242e+02, 1.09549754e+01, 7.86803154e+03, 2.25485839e+03,
         2.23974813e+02],
        [2.98287932e+02, 3.65514402e+00, 8.16623865e+03, 7.78117568e+02,
         3.30487613e+02]], shape=(237312, 5)),
 array([[1738.99062093, 5399.73493232, 3514.55275719],
        [1516.39314729, 4633.5985757 , 4213.09900032],
        [2433.85102081, 6239.34116026, 2270.27371649],
        ...,
        [5573.94801427, 3602.14083553,  613.94336398],
        [5240.74968704, 4656.30913376,  449.80592973],
        [7000.34820584, 1913.20062046,  361.29500126]], shape=(237312, 3)))